# AMML Rank Gated

Лучшее решение, сабмит **58606250**, Public **0,962375 NDCG@10**.

Положите архив соревнования рядом с ноутбуком или укажите его путь в `AMML_DATA`. Ноутбук пересчитывает промежуточные оценки из исходных CSV и обучает стек CatBoost. Для запуска нужны два GPU NVIDIA с 16 ГБ на каждом и доступ к Hugging Face.


In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

data_path = Path(os.environ.get("AMML_DATA", ".")).expanduser().resolve()
raw_names = ("train.csv", "test.csv", "sample_submission.csv")
if data_path.is_dir() and not all((data_path / name).is_file() for name in raw_names):
    data_path /= "amml-ranking-competition-2026.zip"

if not data_path.exists():
    raise FileNotFoundError(
        "Эм а где зип с файлами бро"
    )

feature_dir = TemporaryDirectory(prefix="amml-features-")
root = Path(feature_dir.name)
if data_path.is_dir():
    data = data_path
else:
    data = root
    with ZipFile(data_path) as inputs:
        for name in raw_names:
            (data / name).write_bytes(inputs.read(name))

output_dir = Path(os.environ.get("AMML_OUTPUT", "outputs")).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)


## Признаки и промежуточный стек

Ранжировщики используют предобученные чекпойнты с фиксированными ревизиями. Модели CatBoost обучаются на разметке соревнования.


In [ ]:
import gc

def run_stage(stage, expected):
    print(f"Этап: {stage.__name__}", flush=True)
    stage(data, root)
    missing = [name for name in expected if not (root / name).is_file()]
    if missing:
        raise RuntimeError(f"{stage.__name__} не создал файлы: {missing}")
    gc.collect()
    torch.cuda.empty_cache()

import torch

if torch.cuda.device_count() < 2:
    raise RuntimeError("Запустить на каггле нужно с двумя GPU, а не с одним.")
print("PyTorch:", torch.__version__, "GPU:", [torch.cuda.get_device_name(i) for i in range(2)])

# Семантические оценки BGE и Horizon Small
def semantic_scores(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import torch
    print("CUDA", torch.cuda.is_available(), flush=True)
    if not torch.cuda.is_available():
        raise RuntimeError("This scoring run expects a GPU.")
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value

    cols = ["id", "query", "text", "title"]
    frame = pd.concat([pd.read_csv(data / f"{split}.csv", usecols=cols, keep_default_na=False) for split in ("train", "test")], ignore_index=True)
    frame["query"] = frame["query"].map(clean_text)
    frame["text_clean"] = frame["text"].map(clean_text)
    frame["title_clean"] = frame["title"].map(clean_text)
    import os

    import numpy as np
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer


    CHECKPOINTS = {
        "bge": (
            "BAAI/bge-reranker-v2-m3",
            "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
            320,
        ),
        "horizon": (
            "Horizon-Labs/multilingual-reranker-small",
            "23139fec6f1f448cdf89a5ac2d2c6bf5e8c7b398",
            320,
        ),
    }


    @torch.inference_mode()
    def score_checkpoint(frame, checkpoint, variants=("text", "title_text")):
        model_name, revision, max_length = CHECKPOINTS[checkpoint]
        use_cuda = torch.cuda.is_available()
        device = torch.device("cuda:0" if use_cuda else "cpu")
        if not use_cuda:
            torch.set_num_threads(min(6, os.cpu_count() or 1))
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            revision=revision,
            torch_dtype=torch.float16 if use_cuda else torch.float32,
            trust_remote_code=False,
            attn_implementation="sdpa",
        ).eval().to(device)
        queries = frame["query"].tolist()
        texts = frame["text_clean"].tolist()
        titles = frame["title_clean"].tolist()
        batch_size = 96 if use_cuda else 16
        columns = {}

        for variant in variants:
            passages = texts if variant == "text" else [
                f"{title}. {passage}" if title else passage for title, passage in zip(titles, texts)
            ]
            order = sorted(range(len(frame)), key=lambda i: len(passages[i]))
            scores = np.empty(len(frame), dtype=np.float32)
            for left in range(0, len(order), batch_size):
                batch = order[left:left + batch_size]
                inputs = tokenizer(
                    [queries[i][:400] for i in batch],
                    [passages[i][:3500] for i in batch],
                    padding=True, truncation="only_second", max_length=max_length,
                    pad_to_multiple_of=8, return_tensors="pt",
                )
                logits = model(**inputs.to(device)).logits
                scores[batch] = logits.reshape(-1).float().cpu().numpy()
                if left % (batch_size * 60) == 0:
                    print(f"{checkpoint}/{variant}: {left + len(batch)}/{len(order)}", flush=True)
            if not np.isfinite(scores).all():
                raise ValueError(f"Non-finite scores from {checkpoint}/{variant}")
            columns[f"{checkpoint}_{variant}"] = scores
        del model
        if use_cuda:
            torch.cuda.empty_cache()
        return columns

    for checkpoint, variants in (("bge", ("text", "title_text")), ("horizon", ("title_text",))):
        scored = score_checkpoint(frame, checkpoint, variants)
        np.savez_compressed((root / f"{checkpoint}_scores.npz"), ids=frame["id"].to_numpy(dtype=str), **scored)

run_stage(semantic_scores, ('bge_scores.npz', 'horizon_scores.npz'))


# Оценки Horizon Base
def horizon_base(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    frame = pd.concat([train, test], ignore_index=True)
    import os

    import numpy as np
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer


    CHECKPOINTS = {
        "bge": (
            "BAAI/bge-reranker-v2-m3",
            "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
            320,
        ),
        "horizon": (
            "Horizon-Labs/multilingual-reranker-small",
            "23139fec6f1f448cdf89a5ac2d2c6bf5e8c7b398",
            320,
        ),
    }


    @torch.inference_mode()
    def score_checkpoint(frame, checkpoint, variants=("text", "title_text")):
        model_name, revision, max_length = CHECKPOINTS[checkpoint]
        use_cuda = torch.cuda.is_available()
        device = torch.device("cuda:0" if use_cuda else "cpu")
        if not use_cuda:
            torch.set_num_threads(min(6, os.cpu_count() or 1))
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            revision=revision,
            torch_dtype=torch.float16 if use_cuda else torch.float32,
            trust_remote_code=False,
            attn_implementation="sdpa",
        ).eval().to(device)
        queries = frame["query"].tolist()
        texts = frame["text_clean"].tolist()
        titles = frame["title_clean"].tolist()
        batch_size = 96 if use_cuda else 16
        columns = {}

        for variant in variants:
            passages = texts if variant == "text" else [
                f"{title}. {passage}" if title else passage for title, passage in zip(titles, texts)
            ]
            order = sorted(range(len(frame)), key=lambda i: len(passages[i]))
            scores = np.empty(len(frame), dtype=np.float32)
            for left in range(0, len(order), batch_size):
                batch = order[left:left + batch_size]
                inputs = tokenizer(
                    [queries[i][:400] for i in batch],
                    [passages[i][:3500] for i in batch],
                    padding=True, truncation="only_second", max_length=max_length,
                    pad_to_multiple_of=8, return_tensors="pt",
                )
                logits = model(**inputs.to(device)).logits
                scores[batch] = logits.reshape(-1).float().cpu().numpy()
                if left % (batch_size * 60) == 0:
                    print(f"{checkpoint}/{variant}: {left + len(batch)}/{len(order)}", flush=True)
            if not np.isfinite(scores).all():
                raise ValueError(f"Non-finite scores from {checkpoint}/{variant}")
            columns[f"{checkpoint}_{variant}"] = scores
        del model
        if use_cuda:
            torch.cuda.empty_cache()
        return columns

    CHECKPOINTS = {'horizon_base': ('Horizon-Labs/multilingual-reranker-base', '7330f6d0b3f560af0c2b9a63820ea299a45fd91b', 384)}

    scores = score_checkpoint(frame, 'horizon_base', ("title_text",))
    np.savez_compressed((root / "horizon_base_scores.npz"), ids=frame["id"].to_numpy(dtype=str), **scores)

run_stage(horizon_base, ('horizon_base_scores.npz',))


# Оценки Lamar
def lamar_scores(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    frame = pd.concat([train, test], ignore_index=True)
    import os

    import numpy as np
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer


    CHECKPOINTS = {
        "bge": (
            "BAAI/bge-reranker-v2-m3",
            "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
            320,
        ),
        "horizon": (
            "Horizon-Labs/multilingual-reranker-small",
            "23139fec6f1f448cdf89a5ac2d2c6bf5e8c7b398",
            320,
        ),
    }


    @torch.inference_mode()
    def score_checkpoint(frame, checkpoint, variants=("text", "title_text")):
        model_name, revision, max_length = CHECKPOINTS[checkpoint]
        use_cuda = torch.cuda.is_available()
        device = torch.device("cuda:0" if use_cuda else "cpu")
        if not use_cuda:
            torch.set_num_threads(min(6, os.cpu_count() or 1))
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            revision=revision,
            torch_dtype=torch.float16 if use_cuda else torch.float32,
            trust_remote_code=False,
            attn_implementation="sdpa",
        ).eval().to(device)
        queries = frame["query"].tolist()
        texts = frame["text_clean"].tolist()
        titles = frame["title_clean"].tolist()
        batch_size = 96 if use_cuda else 16
        columns = {}

        for variant in variants:
            passages = texts if variant == "text" else [
                f"{title}. {passage}" if title else passage for title, passage in zip(titles, texts)
            ]
            order = sorted(range(len(frame)), key=lambda i: len(passages[i]))
            scores = np.empty(len(frame), dtype=np.float32)
            for left in range(0, len(order), batch_size):
                batch = order[left:left + batch_size]
                inputs = tokenizer(
                    [queries[i][:400] for i in batch],
                    [passages[i][:3500] for i in batch],
                    padding=True, truncation="only_second", max_length=max_length,
                    pad_to_multiple_of=8, return_tensors="pt",
                )
                logits = model(**inputs.to(device)).logits
                scores[batch] = logits.reshape(-1).float().cpu().numpy()
                if left % (batch_size * 60) == 0:
                    print(f"{checkpoint}/{variant}: {left + len(batch)}/{len(order)}", flush=True)
            if not np.isfinite(scores).all():
                raise ValueError(f"Non-finite scores from {checkpoint}/{variant}")
            columns[f"{checkpoint}_{variant}"] = scores
        del model
        if use_cuda:
            torch.cuda.empty_cache()
        return columns

    CHECKPOINTS = {'lamar': ('nlpai-lab/LAMAR-600m', '3af1f97945392da8ef46217640641355928aeb59', 448)}

    scores = score_checkpoint(frame, 'lamar', ("title_text",))
    np.savez_compressed((root / "lamar_scores.npz"), ids=frame["id"].to_numpy(dtype=str), **scores)

run_stage(lamar_scores, ('lamar_scores.npz',))


# Jina на группах валидации
def jina_validation(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    frame = pd.concat([train, test], ignore_index=True)
    import os

    import numpy as np
    import pandas as pd
    import torch
    from transformers import AutoModel, AutoTokenizer

    JINA = "jinaai/jina-reranker-v3.5"
    JINA_REV = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"


    @torch.inference_mode()
    def jina_scores(frame, max_groups=None):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if device.type == "cpu":
            torch.set_num_threads(4)
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(JINA, revision=JINA_REV, trust_remote_code=False)
        model = AutoModel.from_pretrained(
            JINA, revision=JINA_REV, trust_remote_code=True, torch_dtype=dtype,
            attn_implementation="sdpa", use_safetensors=True,
        ).eval().to(device)
        tokenizer.padding_side = "left"
        model._tokenizer = tokenizer
        groups = list(frame.groupby("query_id", sort=True).indices.values())
        if max_groups is not None:
            selected = np.random.default_rng(41).choice(len(groups), max_groups, replace=False)
            groups = [groups[i] for i in selected]
        scores = np.full((len(frame), 2), np.nan, dtype=np.float32)
        for number, indices in enumerate(groups, 1):
            group = frame.iloc[indices]
            documents = [
                f"{title}. {text[:950]}" if title else text[:950]
                for title, text in zip(group["title_clean"], group["text_clean"])
            ]
            query = group["query"].iloc[0]
            for column, order in enumerate((np.arange(len(group)), np.arange(len(group)-1, -1, -1))):
                output = model.rerank(query, [documents[i] for i in order])
                for row in output:
                    scores[indices[order[row["index"]]], column] = row["relevance_score"]
            if number == 1 or number % 25 == 0:
                print(f"Jina groups: {number}/{len(groups)}", flush=True)
        found = np.isfinite(scores).all(axis=1)
        return frame["id"].to_numpy(dtype=str)[found], scores[found].mean(axis=1)



    from sklearn.model_selection import GroupShuffleSplit
    keep = np.zeros(len(train), dtype=bool)
    for seed in (41,73):
        _, idx = next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=seed).split(train,groups=train.query_id))
        keep[idx] = True
    validation = train.iloc[np.flatnonzero(keep)]
    ids, scores = jina_scores(validation)
    np.savez_compressed((root / "jina_validation_scores.npz"),ids=ids,jina_title_text=scores)

run_stage(jina_validation, ('jina_validation_scores.npz',))


# Jina на тестовых запросах
def jina_test(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    frame = pd.concat([train, test], ignore_index=True)
    import os

    import numpy as np
    import pandas as pd
    import torch
    from transformers import AutoModel, AutoTokenizer

    JINA = "jinaai/jina-reranker-v3.5"
    JINA_REV = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"


    @torch.inference_mode()
    def jina_batch_scores(frame, device_index=0):
        from importlib import import_module

        device = torch.device(f"cuda:{device_index}" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(JINA, revision=JINA_REV, trust_remote_code=False)
        model = AutoModel.from_pretrained(
            JINA, revision=JINA_REV, trust_remote_code=True, torch_dtype=dtype,
            attn_implementation="sdpa", use_safetensors=True,
        ).eval().to(device)
        tokenizer.padding_side = "left"
        format_prompt = import_module(model.__class__.__module__).format_docs_prompts_func
        groups = {}
        queries = frame["query"].tolist()
        docs = [
            f"{title}. {text[:950]}" if title else text[:950]
            for title, text in zip(frame["title_clean"], frame["text_clean"])
        ]
        for positions in frame.groupby("query_id", sort=True).indices.values():
            groups.setdefault(len(positions), []).append(positions)
        scores = np.full((len(frame), 2), np.nan, dtype=np.float32)
        completed = 0
        for size, positions_list in sorted(groups.items()):
            ordered = sorted(positions_list, key=lambda positions: sum(len(docs[i]) for i in positions))
            for start in range(0, len(ordered), 3):
                block = ordered[start:start + 3]
                positions = [row for group in block for row in (group, group[::-1])]
                prompts = [
                    format_prompt(
                        queries[indices[0]], [docs[i] for i in indices],
                        special_tokens=model.special_tokens, no_thinking=True,
                    )
                    for indices in positions
                ]
                inputs = tokenizer(prompts, padding=True, pad_to_multiple_of=8, return_tensors="pt").to(device)
                mask = inputs["attention_mask"]
                position_ids = mask.long().cumsum(-1).sub(1).clamp_min(0)
                # The reference forward pass retains all hidden layers. Its rank head needs only the last.
                last = model.model(**inputs, position_ids=position_ids, use_cache=False).last_hidden_state
                doc_mask = inputs["input_ids"].eq(model.doc_embed_token_id)
                query_mask = inputs["input_ids"].eq(model.query_embed_token_id)
                assert torch.all(doc_mask.sum(1).eq(size)) and torch.all(query_mask.sum(1).eq(1))
                d = model.projector(last[doc_mask].reshape(len(prompts), size, -1))
                q = model.projector(last[query_mask]).unsqueeze(1)
                values = torch.nn.functional.cosine_similarity(d.float(), q.float(), dim=-1).cpu().numpy()
                for j, (indices, value) in enumerate(zip(positions, values)):
                    scores[indices,j%2] = value
                completed += len(block)
                if completed % 75 < len(block):
                    print("Jina groups", completed, flush=True)
        if not np.isfinite(scores).all():
            raise ValueError("Non-finite Jina scores.")
        return scores.mean(axis=1)

    from concurrent.futures import ThreadPoolExecutor
    devices = min(2, torch.cuda.device_count())
    if devices <= 1:
        scores = jina_batch_scores(test)
    else:
        query_codes = pd.factorize(test.query_id, sort=True)[0]
        subsets = [np.flatnonzero(query_codes % devices == gpu) for gpu in range(devices)]
        scores = np.empty(len(test), dtype=np.float32)
        with ThreadPoolExecutor(max_workers=devices) as executor:
            jobs = [executor.submit(jina_batch_scores, test.iloc[rows], gpu) for gpu, rows in enumerate(subsets)]
            for rows, job in zip(subsets, jobs):
                scores[rows] = job.result()
    np.savez_compressed((root / "jina_test_scores.npz"), ids=test["id"].to_numpy(dtype=str), jina_title_text=scores)

run_stage(jina_test, ('jina_test_scores.npz',))


# Jina на двух представлениях текста
def jina_two_views(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    import os

    import numpy as np
    import pandas as pd
    import torch
    from transformers import AutoModel, AutoTokenizer

    JINA = "jinaai/jina-reranker-v3.5"
    JINA_REV = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"


    @torch.inference_mode()
    def jina_batch_scores(frame, device_index=0):
        from importlib import import_module

        device = torch.device(f"cuda:{device_index}" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(JINA, revision=JINA_REV, trust_remote_code=False)
        model = AutoModel.from_pretrained(
            JINA, revision=JINA_REV, trust_remote_code=True, torch_dtype=dtype,
            attn_implementation="sdpa", use_safetensors=True,
        ).eval().to(device)
        tokenizer.padding_side = "left"
        format_prompt = import_module(model.__class__.__module__).format_docs_prompts_func
        groups = {}
        queries = frame["query"].tolist()
        docs = [
            f"{title}. {text}" if title else text
            for title, text in zip(frame["title_clean"], frame["text_clean"])
        ]
        for positions in frame.groupby("query_id", sort=True).indices.values():
            groups.setdefault(len(positions), []).append(positions)
        scores = np.full((len(frame), 2), np.nan, dtype=np.float32)
        completed = 0
        for size, positions_list in sorted(groups.items()):
            ordered = sorted(positions_list, key=lambda positions: sum(len(docs[i]) for i in positions))
            for start in range(0, len(ordered), 2):
                block = ordered[start:start + 2]
                positions = [row for group in block for row in (group, group[::-1])]
                prompts = [
                    format_prompt(
                        queries[indices[0]], [docs[i] for i in indices],
                        special_tokens=model.special_tokens, no_thinking=True,
                    )
                    for indices in positions
                ]
                inputs = tokenizer(prompts, padding=True, pad_to_multiple_of=8, return_tensors="pt").to(device)
                mask = inputs["attention_mask"]
                position_ids = mask.long().cumsum(-1).sub(1).clamp_min(0)
                # The reference forward pass retains all hidden layers. Its rank head needs only the last.
                last = model.model(**inputs, position_ids=position_ids, use_cache=False).last_hidden_state
                doc_mask = inputs["input_ids"].eq(model.doc_embed_token_id)
                query_mask = inputs["input_ids"].eq(model.query_embed_token_id)
                assert torch.all(doc_mask.sum(1).eq(size)) and torch.all(query_mask.sum(1).eq(1))
                d = model.projector(last[doc_mask].reshape(len(prompts), size, -1))
                q = model.projector(last[query_mask]).unsqueeze(1)
                values = torch.nn.functional.cosine_similarity(d.float(), q.float(), dim=-1).cpu().numpy()
                for j, (indices, value) in enumerate(zip(positions, values)):
                    scores[indices,j%2] = value
                completed += len(block)
                if completed % 75 < len(block):
                    print("Jina groups", completed, flush=True)
        if not np.isfinite(scores).all():
            raise ValueError("Non-finite Jina scores.")
        return scores.mean(axis=1)


    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")


    def repair_encoding(value):
        """Repair long Cyrillic runs, leaving surrounding text byte-for-byte intact."""
        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    from concurrent.futures import ThreadPoolExecutor

    from sklearn.model_selection import GroupShuffleSplit


    def query_split(seed):
        _, indices = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[indices]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    audit_queries = set(np.random.default_rng(2026).choice(
        sorted(set(train["query_id"]) - fit_queries), size=350, replace=False
    ))
    frame = pd.concat([train[train["query_id"].isin(fit_queries | audit_queries)], test], ignore_index=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")


    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))


    def query_window(query, text, limit=1250):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


    baseline = frame.copy()
    baseline["text_clean"] = baseline["text_clean"].str[:950]
    windowed = frame.copy()
    windowed["text_clean"] = [
        query_window(query, clean_text(repair_encoding(text)))
        for query, text in zip(frame["query"], frame["text"])
    ]
    # Omit the title in the second view so agreement also tests title sensitivity.
    windowed["title_clean"] = ""
    views = pd.concat([baseline, windowed], keys=["base", "window"])
    views["query_id"] = views.index.get_level_values(0) + ":" + views["query_id"]
    views = views.reset_index(drop=True)

    devices = min(2, torch.cuda.device_count())
    if not devices:
        raise RuntimeError("A Kaggle GPU accelerator is required.")
    query_codes = pd.factorize(views["query_id"], sort=True)[0]
    subsets = [np.flatnonzero(query_codes % devices == gpu) for gpu in range(devices)]
    scores = np.empty(len(views), dtype=np.float32)
    with ThreadPoolExecutor(max_workers=devices) as executor:
        jobs = [executor.submit(jina_batch_scores, views.iloc[rows], gpu) for gpu, rows in enumerate(subsets)]
        for rows, job in zip(subsets, jobs):
            scores[rows] = job.result()
    frame["base"], frame["window"] = np.split(scores, 2)


    def by_query_z(values):
        groups = values.groupby(frame["query_id"])
        return (values - groups.transform("mean")) / groups.transform("std").fillna(0).clip(lower=1e-6)


    sample = pd.read_csv(data / "sample_submission.csv")
    raw_scores = .65 * by_query_z(frame["base"]) + .35 * by_query_z(frame["window"])
    scores = sample["id"].map(pd.Series(raw_scores.to_numpy(), index=frame["id"]))
    assert np.isfinite(scores).all()
    sample.assign(prediction=scores).to_csv(root / "jina_views.csv", index=False)

run_stage(jina_two_views, ('jina_views.csv',))


# Jina на трёх представлениях и признаки
def jina_full_views(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata
    from threading import Lock


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    import os

    import numpy as np
    import pandas as pd
    import torch
    from transformers import AutoModel, AutoTokenizer

    JINA = "jinaai/jina-reranker-v3.5"
    JINA_REV = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"


    MODEL_LOAD_LOCK = Lock()


    @torch.inference_mode()
    def jina_batch_scores(frame, device_index=0):
        from importlib import import_module

        device = torch.device(f"cuda:{device_index}" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        if device.type == "cuda":
            torch.cuda.set_device(device)
        with MODEL_LOAD_LOCK:
            tokenizer = AutoTokenizer.from_pretrained(JINA, revision=JINA_REV, trust_remote_code=False)
            model = AutoModel.from_pretrained(
                JINA, revision=JINA_REV, trust_remote_code=True, torch_dtype=dtype,
                attn_implementation="sdpa", use_safetensors=True,
            )
            # The scoring path calls the backbone and projector directly.
            model.lm_head = torch.nn.Identity()
            model = model.eval().to(device)
        tokenizer.padding_side = "left"
        format_prompt = import_module(model.__class__.__module__).format_docs_prompts_func
        groups = {}
        queries = frame["query"].tolist()
        docs = [
            f"{title}. {text}" if title else text
            for title, text in zip(frame["title_clean"], frame["text_clean"])
        ]
        for positions in frame.groupby("query_id", sort=True).indices.values():
            groups.setdefault(len(positions), []).append(positions)
        scores = np.full((len(frame), 2), np.nan, dtype=np.float32)
        def forward(prompts, size):
            inputs = tokenizer(prompts, padding=True, pad_to_multiple_of=8, return_tensors="pt")
            length = inputs["input_ids"].shape[1]
            if len(prompts) > 1 and len(prompts) * length ** 2 > 36_000_000:
                midpoint = len(prompts) // 2
                return np.concatenate((forward(prompts[:midpoint], size), forward(prompts[midpoint:], size)))
            inputs = inputs.to(device)
            mask = inputs["attention_mask"]
            position_ids = mask.long().cumsum(-1).sub(1).clamp_min(0)
            # The reference forward pass retains all hidden layers. Its rank head needs only the last.
            last = model.model(**inputs, position_ids=position_ids, use_cache=False).last_hidden_state
            doc_mask = inputs["input_ids"].eq(model.doc_embed_token_id)
            query_mask = inputs["input_ids"].eq(model.query_embed_token_id)
            assert torch.all(doc_mask.sum(1).eq(size)) and torch.all(query_mask.sum(1).eq(1))
            d = model.projector(last[doc_mask].reshape(len(prompts), size, -1))
            q = model.projector(last[query_mask]).unsqueeze(1)
            return torch.nn.functional.cosine_similarity(d.float(), q.float(), dim=-1).cpu().numpy()

        completed = 0
        for size, positions_list in sorted(groups.items()):
            ordered = sorted(positions_list, key=lambda positions: sum(len(docs[i]) for i in positions))
            for start in range(0, len(ordered), 2):
                block = ordered[start:start + 2]
                positions = [row for group in block for row in (group, group[::-1])]
                prompts = [
                    format_prompt(
                        queries[indices[0]], [docs[i] for i in indices],
                        special_tokens=model.special_tokens, no_thinking=True,
                    )
                    for indices in positions
                ]
                values = forward(prompts, size)
                for j, (indices, value) in enumerate(zip(positions, values)):
                    scores[indices,j%2] = value
                completed += len(block)
                if completed % 75 < len(block):
                    print("Jina groups", completed, flush=True)
        if not np.isfinite(scores).all():
            raise ValueError("Non-finite Jina scores.")
        return scores.mean(axis=1)



    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
    _MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
    _LATIN = re.compile(r"[A-Za-z]")
    _CYRILLIC = re.compile(r"[А-Яа-яЁё]")
    _LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


    def repair_encoding(value):
        """Reverse a validated whole-passage encoding error or recover individual runs."""
        if _RUN.search(value) is None:
            return value
        try:
            return value.encode("cp1251").decode("utf-8")
        except UnicodeError:
            pass

        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    def repair_lookalikes(value):
        """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
        def restore(match):
            word = match.group()
            latin = _LATIN.findall(word)
            if not latin or len(_CYRILLIC.findall(word)) < 2:
                return word
            if all(ord(letter) in _LOOKALIKES for letter in latin):
                return word.translate(_LOOKALIKES)
            return word

        return _MIXED_WORD.sub(restore, value)
    from concurrent.futures import ThreadPoolExecutor

    from sklearn.model_selection import GroupShuffleSplit


    def cached(name):
        with np.load(next(root.rglob(name)), allow_pickle=False) as source:
            result = pd.DataFrame({key: source[key] for key in source.files if key != "ids"}, index=source["ids"])
        assert result.index.is_unique and np.isfinite(result.to_numpy()).all()
        return result


    def query_split(seed):
        _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[rows]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    remainder = set(train["query_id"]) - fit_queries
    old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
    audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
    validation = train[train["query_id"].isin(fit_queries | audit)]
    frame = pd.concat([validation, test], ignore_index=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")

    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))

    def query_window(query, text, limit):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))

    older = pd.concat([cached(f"jina_{split}_scores.npz") for split in ("validation", "test")])
    frame["base"] = frame["id"].map(older["jina_title_text"])
    baseline = frame[frame["query_id"].isin(audit)].copy()
    baseline["text_clean"] = baseline["text_clean"].str[:950]

    windowed = frame.loc[frame["relevance"].notna()].copy()
    # Match the original positive experiment on labeled data.
    def partial_repair(value):
        def decode(match):
            original = match.group()
            repaired = original.encode("cp1251").decode("utf-8")
            return repaired if all("А" <= char <= "я" or char in "Ёё" for char in repaired) else original
        return _RUN.sub(decode, value)


    windowed["text_clean"] = [
        query_window(query, clean_text(partial_repair(text)), 1250)
        for query, text in zip(windowed["query"], windowed["text"])
    ]
    windowed["title_clean"] = ""

    full = frame.copy()
    full["title_clean"] = full["title_clean"].map(repair_lookalikes)
    full["text_clean"] = [
        query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
        for query, text in zip(full["query"], full["text"])
    ]

    parts = {"base": baseline, "window": windowed, "full": full}
    views = pd.concat(parts, names=["view", "row"]).reset_index(level="view").reset_index(drop=True)
    views["query_id"] = views["view"] + ":" + views["query_id"]
    devices = min(2, torch.cuda.device_count())
    assert devices, "GPU unavailable"
    codes = pd.factorize(views["query_id"], sort=True)[0]
    subsets = [np.flatnonzero(codes % devices == gpu) for gpu in range(devices)]
    scores = np.empty(len(views), dtype=np.float32)
    with ThreadPoolExecutor(max_workers=devices) as executor:
        jobs = [executor.submit(jina_batch_scores, views.iloc[rows], gpu) for gpu, rows in enumerate(subsets)]
        for rows, job in zip(subsets, jobs):
            scores[rows] = job.result()
    views["score"] = scores
    for variant in parts:
        by_id = views.loc[views["view"].eq(variant)].set_index("id")["score"]
        new = frame["id"].map(by_id)
        frame[variant] = new.combine_first(frame[variant]) if variant in frame else new

    def group_z(values):
        group = values.groupby(frame["query_id"])
        return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

    base = group_z(frame["base"])
    win = group_z(frame["window"])
    full_score = group_z(frame["full"])
    old_view = pd.read_csv(next(root.rglob("jina_views.csv"))).set_index("id")["prediction"]
    prior_win = (frame["id"].map(old_view) - .65 * base) / .35
    win = win.combine_first(prior_win)
    assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
    frame[["id", "query_id"]].assign(base=base, window=win, full=full_score).to_csv(
        (root / "full_graph_scores.csv"), index=False
    )

run_stage(jina_full_views, ('full_graph_scores.csv',))


# Обучение стека признаков Jina
def jina_stack(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
    _MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
    _LATIN = re.compile(r"[A-Za-z]")
    _CYRILLIC = re.compile(r"[А-Яа-яЁё]")
    _LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


    def repair_encoding(value):
        """Reverse a validated whole-passage encoding error or recover individual runs."""
        if _RUN.search(value) is None:
            return value
        try:
            return value.encode("cp1251").decode("utf-8")
        except UnicodeError:
            pass

        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    def repair_lookalikes(value):
        """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
        def restore(match):
            word = match.group()
            latin = _LATIN.findall(word)
            if not latin or len(_CYRILLIC.findall(word)) < 2:
                return word
            if all(ord(letter) in _LOOKALIKES for letter in latin):
                return word.translate(_LOOKALIKES)
            return word

        return _MIXED_WORD.sub(restore, value)
    from catboost import CatBoostClassifier
    from sklearn.model_selection import GroupShuffleSplit

    def cached(name):
        with np.load(next(root.rglob(name)), allow_pickle=False) as source:
            result = pd.DataFrame({key: source[key] for key in source.files if key != "ids"}, index=source["ids"])
        assert result.index.is_unique and np.isfinite(result.to_numpy()).all()
        return result


    def query_split(seed):
        _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[rows]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    remainder = set(train["query_id"]) - fit_queries
    old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
    audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
    validation = train[train["query_id"].isin(fit_queries | audit)]
    frame = pd.concat([validation, test], ignore_index=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")

    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))

    def query_window(query, text, limit):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


    graph = pd.read_csv(next(root.rglob("full_graph_scores.csv"))).set_index("id", verify_integrity=True)
    base = frame["id"].map(graph["base"])
    win = frame["id"].map(graph["window"])
    full_score = frame["id"].map(graph["full"])
    assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
    full = frame.copy()
    full["title_clean"] = full["title_clean"].map(repair_lookalikes)
    full["text_clean"] = [
        query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
        for query, text in zip(full["query"], full["text"])
    ]

    def group_z(values):
        group = values.groupby(frame["query_id"])
        return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

    semantic = pd.concat([cached(filename) for filename in (
        "bge_scores.npz", "horizon_scores.npz", "horizon_base_scores.npz", "lamar_scores.npz"
    )], axis=1)
    assert semantic.index.is_unique
    features = pd.DataFrame({"base": base, "window": win, "full": full_score, "title_empty": frame["title_clean"].eq("")})
    for name in semantic:
        feature = frame["id"].map(semantic[name])
        features[name] = group_z(feature)
        features[f"{name}_raw"] = feature
    for name, value in (("full_minus_window", full_score - win), ("full_minus_base", full_score - base)):
        features[name] = value
    features["words"] = full["text_clean"].map(lambda text: len(tokens(text)))
    features["title_overlap"] = [
        len(set(tokens(q)).intersection(tokens(t))) for q, t in zip(full["query"], full["title_clean"])
    ]

    def predict(query_ids):
        training = frame["query_id"].isin(query_ids)
        model = CatBoostClassifier(iterations=240, depth=3, learning_rate=.04, l2_leaf_reg=6,
                                   random_seed=42, verbose=False, thread_count=4, allow_writing_files=False)
        model.fit(features.loc[training], frame.loc[training, "relevance"].astype(int))
        stack = group_z(pd.Series(model.predict_proba(features)[:, 1], index=frame.index))
        return pd.DataFrame({
            "full_half": .18 * base + .18 * win + .36 * full_score + .28 * stack,
        })

    predictions = predict(fit_queries)
    sample = pd.read_csv(data / "sample_submission.csv")
    scores_by_id = frame[["id"]].join(predictions).set_index("id", verify_integrity=True)
    scores = sample["id"].map(scores_by_id["full_half"])
    assert np.isfinite(scores).all()
    sample.assign(prediction=scores).to_csv(root / "jina_full_half.csv", index=False)

run_stage(jina_stack, ('jina_full_half.csv',))


# Смесь стека Jina и оценок Qwen 8B
def qwen8_blend(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
    _MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
    _LATIN = re.compile(r"[A-Za-z]")
    _CYRILLIC = re.compile(r"[А-Яа-яЁё]")
    _LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


    def repair_encoding(value):
        """Reverse a validated whole-passage encoding error or recover individual runs."""
        if _RUN.search(value) is None:
            return value
        try:
            return value.encode("cp1251").decode("utf-8")
        except UnicodeError:
            pass

        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    def repair_lookalikes(value):
        """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
        def restore(match):
            word = match.group()
            latin = _LATIN.findall(word)
            if not latin or len(_CYRILLIC.findall(word)) < 2:
                return word
            if all(ord(letter) in _LOOKALIKES for letter in latin):
                return word.translate(_LOOKALIKES)
            return word

        return _MIXED_WORD.sub(restore, value)
    from sklearn.model_selection import GroupShuffleSplit



    def query_split(seed):
        _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[rows]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    remainder = set(train["query_id"]) - fit_queries
    old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
    audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
    validation = train[train["query_id"].isin(fit_queries | audit)]
    frame = pd.concat([validation, test], ignore_index=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")

    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))

    def query_window(query, text, limit):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


    graph = pd.read_csv(next(root.rglob("full_graph_scores.csv"))).set_index("id", verify_integrity=True)
    base = frame["id"].map(graph["base"])
    win = frame["id"].map(graph["window"])
    full_score = frame["id"].map(graph["full"])
    assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
    full = frame.copy()
    full["title_clean"] = full["title_clean"].map(repair_lookalikes)
    full["text_clean"] = [
        query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
        for query, text in zip(full["query"], full["text"])
    ]

    def group_z(values):
        group = values.groupby(frame["query_id"])
        return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    NAME = "Qwen/Qwen3-Reranker-8B"
    revision = "77d193c791ed757ca307ee72715aa132723da912"
    print("Model revision:", revision, flush=True)
    tokenizer = AutoTokenizer.from_pretrained(NAME, revision=revision, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
        NAME, revision=revision, dtype=torch.float16, device_map="balanced",
        max_memory={gpu: "13GiB" for gpu in range(torch.cuda.device_count())},
        attn_implementation="sdpa", use_safetensors=True,
    ).eval()
    print("Device map:", model.hf_device_map, flush=True)
    assert all(device not in ("cpu", "disk") for device in model.hf_device_map.values())

    prefix = tokenizer.encode(
        '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
        'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n',
        add_special_tokens=False,
    )
    suffix = tokenizer.encode(
        "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False,
    )
    yes, no = (tokenizer.convert_tokens_to_ids(word) for word in ("yes", "no"))

    @torch.inference_mode()
    def rank(rows):
        prompts = [
            "<Instruct>: Given a Russian question, retrieve passages that directly answer it or provide evidence needed to answer it.\n"
            f"<Query>: {query}\n<Document>: {title}. {passage}" if title else
            "<Instruct>: Given a Russian question, retrieve passages that directly answer it or provide evidence needed to answer it.\n"
            f"<Query>: {query}\n<Document>: {passage}"
            for query, title, passage in zip(rows["query"], rows["title_clean"], rows["text_clean"])
        ]
        order = sorted(range(len(rows)), key=lambda index: len(prompts[index]))
        values = np.empty(len(rows), dtype=np.float32)
        for start in range(0, len(order), 16):
            batch = order[start:start + 16]
            encoded = tokenizer([prompts[i] for i in batch], add_special_tokens=False,
                                truncation=True, max_length=640 - len(prefix) - len(suffix))
            inputs = tokenizer.pad({"input_ids": [prefix + ids + suffix for ids in encoded["input_ids"]]},
                                   padding=True, pad_to_multiple_of=8, return_tensors="pt")
            logits = model(**inputs.to(model.get_input_embeddings().weight.device),
                           use_cache=False, logits_to_keep=1).logits[:, -1]
            values[batch] = (logits[:, yes].float() - logits[:, no].float()).cpu().numpy()
            if start % 1024 == 0:
                print("Scored", start + len(batch), "of", len(rows), flush=True)
        assert np.isfinite(values).all()
        return pd.Series(values, index=rows.index)

    labeled = frame["relevance"].notna()
    selected = .20
    test_qwen = group_z(rank(full.loc[~labeled]).reindex(frame.index))
    sample = pd.read_csv(data / "sample_submission.csv")
    previous = pd.read_csv(next(root.rglob("jina_full_half.csv"))).set_index("id", verify_integrity=True)["prediction"]
    original = group_z(frame["id"].map(previous))
    blend = (1 - selected) * original + selected * test_qwen
    values = sample["id"].map(pd.Series(blend.to_numpy(), index=frame["id"]))
    assert np.isfinite(values).all()
    sample.assign(prediction=values).to_csv((root / "qwen8_submission.csv"), index=False)

run_stage(qwen8_blend, ('qwen8_submission.csv',))


# Оценки Qwen 8B для обучения ранкера
def qwen8_training_scores(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
    _MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
    _LATIN = re.compile(r"[A-Za-z]")
    _CYRILLIC = re.compile(r"[А-Яа-яЁё]")
    _LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


    def repair_encoding(value):
        """Reverse a validated whole-passage encoding error or recover individual runs."""
        if _RUN.search(value) is None:
            return value
        try:
            return value.encode("cp1251").decode("utf-8")
        except UnicodeError:
            pass

        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    def repair_lookalikes(value):
        """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
        def restore(match):
            word = match.group()
            latin = _LATIN.findall(word)
            if not latin or len(_CYRILLIC.findall(word)) < 2:
                return word
            if all(ord(letter) in _LOOKALIKES for letter in latin):
                return word.translate(_LOOKALIKES)
            return word

        return _MIXED_WORD.sub(restore, value)
    from sklearn.model_selection import GroupShuffleSplit



    def query_split(seed):
        _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[rows]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    remainder = set(train["query_id"]) - fit_queries
    old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
    audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
    validation = train[train["query_id"].isin(fit_queries | audit)]
    frame = pd.concat([validation, test], ignore_index=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")

    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))

    def query_window(query, text, limit):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


    graph = pd.read_csv(next(root.rglob("full_graph_scores.csv"))).set_index("id", verify_integrity=True)
    base = frame["id"].map(graph["base"])
    win = frame["id"].map(graph["window"])
    full_score = frame["id"].map(graph["full"])
    assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
    full = frame.copy()
    full["title_clean"] = full["title_clean"].map(repair_lookalikes)
    full["text_clean"] = [
        query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
        for query, text in zip(full["query"], full["text"])
    ]

    def group_z(values):
        group = values.groupby(frame["query_id"])
        return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    NAME = "Qwen/Qwen3-Reranker-8B"
    revision = "77d193c791ed757ca307ee72715aa132723da912"
    print("Model revision:", revision, flush=True)
    tokenizer = AutoTokenizer.from_pretrained(NAME, revision=revision, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
        NAME, revision=revision, dtype=torch.float16, device_map="balanced",
        max_memory={gpu: "13GiB" for gpu in range(torch.cuda.device_count())},
        attn_implementation="sdpa", use_safetensors=True,
    ).eval()
    print("Device map:", model.hf_device_map, flush=True)
    assert all(device not in ("cpu", "disk") for device in model.hf_device_map.values())

    prefix = tokenizer.encode(
        '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
        'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n',
        add_special_tokens=False,
    )
    suffix = tokenizer.encode(
        "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False,
    )
    yes, no = (tokenizer.convert_tokens_to_ids(word) for word in ("yes", "no"))

    @torch.inference_mode()
    def rank(rows):
        prompts = [
            "<Instruct>: Given a Russian question, retrieve passages that directly answer it or provide evidence needed to answer it.\n"
            f"<Query>: {query}\n<Document>: {title}. {passage}" if title else
            "<Instruct>: Given a Russian question, retrieve passages that directly answer it or provide evidence needed to answer it.\n"
            f"<Query>: {query}\n<Document>: {passage}"
            for query, title, passage in zip(rows["query"], rows["title_clean"], rows["text_clean"])
        ]
        order = sorted(range(len(rows)), key=lambda index: len(prompts[index]))
        values = np.empty(len(rows), dtype=np.float32)
        for start in range(0, len(order), 16):
            batch = order[start:start + 16]
            encoded = tokenizer([prompts[i] for i in batch], add_special_tokens=False,
                                truncation=True, max_length=640 - len(prefix) - len(suffix))
            inputs = tokenizer.pad({"input_ids": [prefix + ids + suffix for ids in encoded["input_ids"]]},
                                   padding=True, pad_to_multiple_of=8, return_tensors="pt")
            logits = model(**inputs.to(model.get_input_embeddings().weight.device),
                           use_cache=False, logits_to_keep=1).logits[:, -1]
            values[batch] = (logits[:, yes].float() - logits[:, no].float()).cpu().numpy()
            if start % 1024 == 0:
                print("Scored", start + len(batch), "of", len(rows), flush=True)
        assert np.isfinite(values).all()
        return pd.Series(values, index=rows.index)



    labeled = frame["relevance"].notna()
    qwen = group_z(rank(full.loc[labeled]).reindex(frame.index))
    frame.loc[labeled, ["id","query_id"]].assign(qwen8=qwen).to_csv((root / "qwen8_validation_scores.csv"), index=False)

run_stage(qwen8_training_scores, ('qwen8_validation_scores.csv',))


# Оценки Jina и Qwen 8B на отложенных запросах
def fresh_holdout(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import torch
    print("GPU", torch.cuda.get_device_name(0), "Torch", torch.__version__, flush=True)
    import html
    import re
    import unicodedata
    from threading import Lock


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    import os

    import numpy as np
    import pandas as pd
    import torch
    from transformers import AutoModel, AutoTokenizer

    JINA = "jinaai/jina-reranker-v3.5"
    JINA_REV = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"


    MODEL_LOAD_LOCK = Lock()


    @torch.inference_mode()
    def jina_batch_scores(frame, device_index=0):
        from importlib import import_module

        device = torch.device(f"cuda:{device_index}" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        if device.type == "cuda":
            torch.cuda.set_device(device)
        with MODEL_LOAD_LOCK:
            tokenizer = AutoTokenizer.from_pretrained(JINA, revision=JINA_REV, trust_remote_code=False)
            model = AutoModel.from_pretrained(
                JINA, revision=JINA_REV, trust_remote_code=True, torch_dtype=dtype,
                attn_implementation="sdpa", use_safetensors=True,
            )
            # The scoring path calls the backbone and projector directly.
            model.lm_head = torch.nn.Identity()
            model = model.eval().to(device)
        tokenizer.padding_side = "left"
        format_prompt = import_module(model.__class__.__module__).format_docs_prompts_func
        groups = {}
        queries = frame["query"].tolist()
        docs = [
            f"{title}. {text}" if title else text
            for title, text in zip(frame["title_clean"], frame["text_clean"])
        ]
        for positions in frame.groupby("query_id", sort=True).indices.values():
            groups.setdefault(len(positions), []).append(positions)
        scores = np.full((len(frame), 2), np.nan, dtype=np.float32)
        def forward(prompts, size):
            inputs = tokenizer(prompts, padding=True, pad_to_multiple_of=8, return_tensors="pt")
            length = inputs["input_ids"].shape[1]
            if len(prompts) > 1 and len(prompts) * length ** 2 > 36_000_000:
                midpoint = len(prompts) // 2
                return np.concatenate((forward(prompts[:midpoint], size), forward(prompts[midpoint:], size)))
            inputs = inputs.to(device)
            mask = inputs["attention_mask"]
            position_ids = mask.long().cumsum(-1).sub(1).clamp_min(0)
            # The reference forward pass retains all hidden layers. Its rank head needs only the last.
            last = model.model(**inputs, position_ids=position_ids, use_cache=False).last_hidden_state
            doc_mask = inputs["input_ids"].eq(model.doc_embed_token_id)
            query_mask = inputs["input_ids"].eq(model.query_embed_token_id)
            assert torch.all(doc_mask.sum(1).eq(size)) and torch.all(query_mask.sum(1).eq(1))
            d = model.projector(last[doc_mask].reshape(len(prompts), size, -1))
            q = model.projector(last[query_mask]).unsqueeze(1)
            return torch.nn.functional.cosine_similarity(d.float(), q.float(), dim=-1).cpu().numpy()

        completed = 0
        for size, positions_list in sorted(groups.items()):
            ordered = sorted(positions_list, key=lambda positions: sum(len(docs[i]) for i in positions))
            for start in range(0, len(ordered), 2):
                block = ordered[start:start + 2]
                positions = [row for group in block for row in (group, group[::-1])]
                prompts = [
                    format_prompt(
                        queries[indices[0]], [docs[i] for i in indices],
                        special_tokens=model.special_tokens, no_thinking=True,
                    )
                    for indices in positions
                ]
                values = forward(prompts, size)
                for j, (indices, value) in enumerate(zip(positions, values)):
                    scores[indices,j%2] = value
                completed += len(block)
                if completed % 75 < len(block):
                    print("Jina groups", completed, flush=True)
        if not np.isfinite(scores).all():
            raise ValueError("Non-finite Jina scores.")
        return scores.mean(axis=1)



    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
    _MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
    _LATIN = re.compile(r"[A-Za-z]")
    _CYRILLIC = re.compile(r"[А-Яа-яЁё]")
    _LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


    def repair_encoding(value):
        """Reverse a validated whole-passage encoding error or recover individual runs."""
        if _RUN.search(value) is None:
            return value
        try:
            return value.encode("cp1251").decode("utf-8")
        except UnicodeError:
            pass

        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    def repair_lookalikes(value):
        """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
        def restore(match):
            word = match.group()
            latin = _LATIN.findall(word)
            if not latin or len(_CYRILLIC.findall(word)) < 2:
                return word
            if all(ord(letter) in _LOOKALIKES for letter in latin):
                return word.translate(_LOOKALIKES)
            return word

        return _MIXED_WORD.sub(restore, value)
    from concurrent.futures import ThreadPoolExecutor

    from sklearn.model_selection import GroupShuffleSplit




    def query_split(seed):
        _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[rows]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    remainder = set(train["query_id"]) - fit_queries
    old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
    audit_2027 = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
    left = remainder - old_audit - audit_2027
    audit_2028 = set(np.random.default_rng(2028).choice(sorted(left), size=350, replace=False))
    audit = set(np.random.default_rng(2029).choice(sorted(left - audit_2028), size=350, replace=False))
    frame = train.loc[train["query_id"].isin(audit)].reset_index(drop=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")

    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))

    def query_window(query, text, limit):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


    def repair_runs(value):
        def fix(match):
            original = match.group()
            decoded = original.encode("cp1251").decode("utf-8")
            return decoded if all("А" <= c <= "я" or c in "Ёё" for c in decoded) else original
        return _RUN.sub(fix, value)

    baseline = frame.copy()
    baseline["text_clean"] = baseline["text_clean"].str[:950]
    windowed = frame.copy()
    windowed["title_clean"] = ""
    windowed["text_clean"] = [
        query_window(q, clean_text(repair_runs(t)), 1250)
        for q,t in zip(frame["query"], frame["text"])
    ]
    long_text = frame.copy()
    long_text["title_clean"] = long_text["title_clean"].map(repair_lookalikes)
    long_text["text_clean"] = [
        query_window(q, clean_text(repair_lookalikes(repair_encoding(t))), 1800)
        for q,t in zip(frame["query"], frame["text"])
    ]
    views = pd.concat({"base": baseline, "window": windowed, "full": long_text}, names=["view", "row"])
    views = views.reset_index(level="view").reset_index(drop=True)
    views["query_id"] = views["view"] + ":" + views["query_id"]
    devices = min(2, torch.cuda.device_count())
    assert devices
    codes = pd.factorize(views["query_id"], sort=True)[0]
    subsets = [np.flatnonzero(codes % devices == gpu) for gpu in range(devices)]
    scores = np.empty(len(views), dtype=np.float32)
    with ThreadPoolExecutor(max_workers=devices) as executor:
        futures = [executor.submit(jina_batch_scores, views.iloc[rows], gpu) for gpu, rows in enumerate(subsets)]
        for rows, job in zip(subsets, futures):
            scores[rows] = job.result()
    views["score"] = scores

    def group_z(values):
        by_query = values.groupby(frame["query_id"])
        return (values - by_query.transform("mean")) / by_query.transform("std").fillna(0).clip(lower=1e-6)

    output = frame[["id", "query_id"]].copy()
    for variant in ("base", "window", "full"):
        by_id = views.loc[views["view"].eq(variant)].set_index("id", verify_integrity=True)["score"]
        output[variant] = group_z(frame["id"].map(by_id))
    assert np.isfinite(output[["base", "window","full"]]).all().all()
    for gpu in range(devices):
        with torch.cuda.device(gpu):
            torch.cuda.empty_cache()

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    NAME = "Qwen/Qwen3-Reranker-8B"
    revision = "77d193c791ed757ca307ee72715aa132723da912"
    print("Model revision:", revision, flush=True)
    tokenizer = AutoTokenizer.from_pretrained(NAME, revision=revision, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
        NAME, revision=revision, dtype=torch.float16, device_map="balanced",
        max_memory={gpu: "13GiB" for gpu in range(torch.cuda.device_count())},
        attn_implementation="sdpa", use_safetensors=True,
    ).eval()
    print("Device map:", model.hf_device_map, flush=True)
    assert all(device not in ("cpu", "disk") for device in model.hf_device_map.values())

    prefix = tokenizer.encode(
        '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
        'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n',
        add_special_tokens=False,
    )
    suffix = tokenizer.encode(
        "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False,
    )
    yes, no = (tokenizer.convert_tokens_to_ids(word) for word in ("yes", "no"))

    @torch.inference_mode()
    def rank(rows):
        prompts = [
            "<Instruct>: Given a Russian question, retrieve passages that directly answer it or provide evidence needed to answer it.\n"
            f"<Query>: {query}\n<Document>: {title}. {passage}" if title else
            "<Instruct>: Given a Russian question, retrieve passages that directly answer it or provide evidence needed to answer it.\n"
            f"<Query>: {query}\n<Document>: {passage}"
            for query, title, passage in zip(rows["query"], rows["title_clean"], rows["text_clean"])
        ]
        order = sorted(range(len(rows)), key=lambda index: len(prompts[index]))
        values = np.empty(len(rows), dtype=np.float32)
        for start in range(0, len(order), 16):
            batch = order[start:start + 16]
            encoded = tokenizer([prompts[i] for i in batch], add_special_tokens=False,
                                truncation=True, max_length=640 - len(prefix) - len(suffix))
            inputs = tokenizer.pad({"input_ids": [prefix + ids + suffix for ids in encoded["input_ids"]]},
                                   padding=True, pad_to_multiple_of=8, return_tensors="pt")
            logits = model(**inputs.to(model.get_input_embeddings().weight.device),
                           use_cache=False, logits_to_keep=1).logits[:, -1]
            values[batch] = (logits[:, yes].float() - logits[:, no].float()).cpu().numpy()
            if start % 1024 == 0:
                print("Scored", start + len(batch), "of", len(rows), flush=True)
        assert np.isfinite(values).all()
        return pd.Series(values, index=rows.index)

    output["qwen8"] = group_z(rank(long_text))
    assert np.isfinite(output["qwen8"]).all()
    output.to_csv((root / "fresh_qwen8_scores.csv"), index=False)

run_stage(fresh_holdout, ('fresh_qwen8_scores.csv',))


# Обучение промежуточного попарного ранкера CatBoost
def robust_ranking(data, root):
    import os
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import pandas as pd
    import numpy as np
    import html
    import re
    import unicodedata


    TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
    SPACES = re.compile(r"\s+")
    INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
    MISSING = {"", "null", "none", "nan", "-", "n/a"}


    def clean_text(value):
        value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
        value = TAG.sub(" ", value.translate(INVISIBLE))
        value = SPACES.sub(" ", value).strip()
        return "" if value.lower() in MISSING else value


    def load_split(split):
        frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
        frame["query"] = frame["query"].map(clean_text)
        frame["text_clean"] = frame["text"].map(clean_text)
        frame["title_clean"] = frame["title"].map(clean_text)
        return frame

    train, test = load_split("train"), load_split("test")
    """Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

    import re


    def _continuation_characters():
        characters = []
        for byte in range(0x80, 0xC0):
            try:
                characters.append(bytes([byte]).decode("cp1251"))
            except UnicodeDecodeError:
                continue
        return re.escape("".join(characters))


    _PAIR = rf"[РС][{_continuation_characters()}]"
    _RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
    _MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
    _LATIN = re.compile(r"[A-Za-z]")
    _CYRILLIC = re.compile(r"[А-Яа-яЁё]")
    _LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


    def repair_encoding(value):
        """Reverse a validated whole-passage encoding error or recover individual runs."""
        if _RUN.search(value) is None:
            return value
        try:
            return value.encode("cp1251").decode("utf-8")
        except UnicodeError:
            pass

        def decode(match):
            original = match.group()
            candidate = original.encode("cp1251").decode("utf-8")
            return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

        return _RUN.sub(decode, value)


    def repair_lookalikes(value):
        """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
        def restore(match):
            word = match.group()
            latin = _LATIN.findall(word)
            if not latin or len(_CYRILLIC.findall(word)) < 2:
                return word
            if all(ord(letter) in _LOOKALIKES for letter in latin):
                return word.translate(_LOOKALIKES)
            return word

        return _MIXED_WORD.sub(restore, value)
    from catboost import CatBoostClassifier
    from sklearn.model_selection import GroupShuffleSplit

    def cached(name):
        with np.load(next(root.rglob(name)), allow_pickle=False) as source:
            result = pd.DataFrame({key: source[key] for key in source.files if key != "ids"}, index=source["ids"])
        assert result.index.is_unique and np.isfinite(result.to_numpy()).all()
        return result


    def query_split(seed):
        _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
            train, groups=train["query_id"]
        ))
        return set(train.iloc[rows]["query_id"])


    folds = {seed: query_split(seed) for seed in (41, 73)}
    fit_queries = set.union(*folds.values())
    remainder = set(train["query_id"]) - fit_queries
    old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
    audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
    fresh = pd.read_csv(next(root.rglob("fresh_qwen8_scores.csv")))
    fresh_queries = set(fresh["query_id"])
    assert len(fresh_queries) == 350 and fresh_queries.isdisjoint(fit_queries | audit | old_audit)
    validation = train[train["query_id"].isin(fit_queries | audit | fresh_queries)]
    frame = pd.concat([validation, test], ignore_index=True)

    TOKEN = re.compile(r"[^\W_]+")
    SENTENCE = re.compile(r"(?<=[.!?])\s+")

    def tokens(text):
        return TOKEN.findall(text.casefold().replace("ё", "е"))

    def query_window(query, text, limit):
        if len(text) <= limit:
            return text
        terms = {word[:5] for word in tokens(query) if len(word) >= 3}
        if not terms:
            return text[:limit]
        sentences = SENTENCE.split(text)
        offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
        candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
        candidates.append(len(text) - limit)
        windows = [text[start:start + limit] for start in candidates]
        return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


    original_graph = pd.read_csv(next(root.rglob("full_graph_scores.csv")))
    graph = pd.concat([original_graph, fresh], ignore_index=True).set_index("id", verify_integrity=True)
    base = frame["id"].map(graph["base"])
    win = frame["id"].map(graph["window"])
    full_score = frame["id"].map(graph["full"])
    assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
    full = frame.copy()
    full["title_clean"] = full["title_clean"].map(repair_lookalikes)
    full["text_clean"] = [
        query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
        for query, text in zip(full["query"], full["text"])
    ]

    def group_z(values):
        group = values.groupby(frame["query_id"])
        return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

    semantic = pd.concat([cached(filename) for filename in (
        "bge_scores.npz", "horizon_scores.npz", "horizon_base_scores.npz", "lamar_scores.npz"
    )], axis=1)
    assert semantic.index.is_unique
    features = pd.DataFrame({"base": base, "window": win, "full": full_score, "title_empty": frame["title_clean"].eq("")})
    for name in semantic:
        feature = frame["id"].map(semantic[name])
        features[name] = group_z(feature)
        features[f"{name}_raw"] = feature
    for name, value in (("full_minus_window", full_score - win), ("full_minus_base", full_score - base)):
        features[name] = value
    features["words"] = full["text_clean"].map(lambda text: len(tokens(text)))
    features["title_overlap"] = [
        len(set(tokens(q)).intersection(tokens(t))) for q, t in zip(full["query"], full["title_clean"])
    ]

    def predict(query_ids):
        training = frame["query_id"].isin(query_ids)
        model = CatBoostClassifier(iterations=240, depth=3, learning_rate=.04, l2_leaf_reg=6,
                                   random_seed=42, verbose=False, thread_count=4, allow_writing_files=False)
        model.fit(features.loc[training], frame.loc[training, "relevance"].astype(int))
        stack = group_z(pd.Series(model.predict_proba(features)[:, 1], index=frame.index))
        return pd.DataFrame({
            "full_half": .18 * base + .18 * win + .36 * full_score + .28 * stack,
        })

    from catboost import CatBoostRanker, Pool

    labeled = frame["relevance"].notna()
    old_qwen = pd.read_csv(next(root.rglob("qwen8_validation_scores.csv"))).set_index("id", verify_integrity=True)["qwen8"]
    all_qwen = pd.concat([old_qwen, fresh.set_index("id")["qwen8"]], verify_integrity=True)
    qwen = frame["id"].map(all_qwen)
    previous = pd.read_csv(next(root.rglob("qwen8_submission.csv"))).set_index("id", verify_integrity=True)["prediction"]
    old_jina = pd.read_csv(next(root.rglob("jina_full_half.csv"))).set_index("id", verify_integrity=True)["prediction"]
    deployed = frame["id"].map(previous)
    qwen_test = (deployed - .8 * group_z(frame["id"].map(old_jina))) / .2
    qwen = qwen.combine_first(qwen_test)
    assert np.isfinite(qwen).all()
    matrix = features.assign(
        qwen=qwen,
        qwen_rank=qwen.groupby(frame["query_id"]).rank(pct=True),
        qwen_gap=qwen.groupby(frame["query_id"]).transform("max") - qwen,
        jina_qwen_agreement=qwen - .25 * base - .25 * win - .50 * full_score,
        jina_views_spread=features[["base", "window", "full"]].std(axis=1),
    )

    def run_model(queries):
        ordered = frame.loc[frame["query_id"].isin(queries)].sort_values(["query_id","id"]).index
        group = frame.loc[ordered, "query_id"]
        labels = frame.loc[ordered, "relevance"].astype(int)
        original = .8 * group_z(predict(queries)["full_half"]) + .2 * qwen
        original = deployed.combine_first(original)
        ranker = CatBoostRanker(iterations=320, depth=3, learning_rate=.035, l2_leaf_reg=8,
                                  loss_function="PairLogit", random_seed=42, thread_count=4,
                                  verbose=False, allow_writing_files=False)
        ranker.fit(Pool(matrix.loc[ordered], label=labels, group_id=group))
        rank = group_z(pd.Series(ranker.predict(matrix), index=frame.index))
        return .65 * group_z(original) + .35 * rank

    predictions = run_model(fit_queries)
    sample = pd.read_csv(data / "sample_submission.csv")
    values = sample["id"].map(pd.Series(predictions.to_numpy(), index=frame["id"]))
    assert np.isfinite(values).all()
    sample.assign(prediction=values).to_csv((root / "robust_rank_submission.csv"), index=False)

run_stage(robust_ranking, ('robust_rank_submission.csv',))


## Финальный CatBoost и проверка

Для итогового сабмита используется конфигурация `gated_medium`. Отчёт показывает её качество на двух dev-разбиениях и отложенных запросах. Последняя проверка сравнивает порядок отрывков с лучшим сабмитом.


In [ ]:
import pandas as pd
import numpy as np
import html
import re
import unicodedata


TAG = re.compile(r"<(?:/?[a-z][a-z0-9:-]*)(?:\s+[^<>]{0,300})?\s*/?>", re.I)
SPACES = re.compile(r"\s+")
INVISIBLE = dict.fromkeys(map(ord, "\u00ad\u0300\u0301\u200b\u200c\u200d\u2060\ufeff"))
MISSING = {"", "null", "none", "nan", "-", "n/a"}


def clean_text(value):
    value = unicodedata.normalize("NFKC", html.unescape(html.unescape(value)))
    value = TAG.sub(" ", value.translate(INVISIBLE))
    value = SPACES.sub(" ", value).strip()
    return "" if value.lower() in MISSING else value

def load_split(split):
    frame = pd.read_csv(data / f"{split}.csv", keep_default_na=False)
    frame["query"] = frame["query"].map(clean_text)
    frame["text_clean"] = frame["text"].map(clean_text)
    frame["title_clean"] = frame["title"].map(clean_text)
    return frame

train, test = load_split("train"), load_split("test")
"""Repair detectable UTF-8 Cyrillic runs misdecoded as Windows-1251."""

import re


def _continuation_characters():
    characters = []
    for byte in range(0x80, 0xC0):
        try:
            characters.append(bytes([byte]).decode("cp1251"))
        except UnicodeDecodeError:
            continue
    return re.escape("".join(characters))


_PAIR = rf"[РС][{_continuation_characters()}]"
_RUN = re.compile(rf"(?:{_PAIR}){{3,}}")
_MIXED_WORD = re.compile(r"\b[А-Яа-яЁёA-Za-z]{4,}\b")
_LATIN = re.compile(r"[A-Za-z]")
_CYRILLIC = re.compile(r"[А-Яа-яЁё]")
_LOOKALIKES = str.maketrans("ABCEHKMOPTXYacekopxy", "АВСЕНКМОРТХУасекорху")


def repair_encoding(value):
    """Reverse a validated whole-passage encoding error or recover individual runs."""
    if _RUN.search(value) is None:
        return value
    try:
        return value.encode("cp1251").decode("utf-8")
    except UnicodeError:
        pass

    def decode(match):
        original = match.group()
        candidate = original.encode("cp1251").decode("utf-8")
        return candidate if all("А" <= char <= "я" or char in "Ёё" for char in candidate) else original

    return _RUN.sub(decode, value)


def repair_lookalikes(value):
    """Restore mixed Cyrillic words when every Latin letter has a Cyrillic lookalike."""
    def restore(match):
        word = match.group()
        latin = _LATIN.findall(word)
        if not latin or len(_CYRILLIC.findall(word)) < 2:
            return word
        if all(ord(letter) in _LOOKALIKES for letter in latin):
            return word.translate(_LOOKALIKES)
        return word

    return _MIXED_WORD.sub(restore, value)


from catboost import CatBoostClassifier
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit

def cached(name):
    with np.load(next(root.rglob(name)), allow_pickle=False) as source:
        result = pd.DataFrame({key: source[key] for key in source.files if key != "ids"}, index=source["ids"])
    assert result.index.is_unique and np.isfinite(result.to_numpy()).all()
    return result


def query_split(seed):
    _, rows = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed).split(
        train, groups=train["query_id"]
    ))
    return set(train.iloc[rows]["query_id"])


folds = {seed: query_split(seed) for seed in (41, 73)}
fit_queries = set.union(*folds.values())
remainder = set(train["query_id"]) - fit_queries
old_audit = set(np.random.default_rng(2026).choice(sorted(remainder), size=350, replace=False))
audit = set(np.random.default_rng(2027).choice(sorted(remainder - old_audit), size=350, replace=False))
fresh = pd.read_csv(next(root.rglob("fresh_qwen8_scores.csv")))
fresh_queries = set(fresh["query_id"])
assert len(fresh_queries) == 350 and fresh_queries.isdisjoint(fit_queries | audit | old_audit)
validation = train[train["query_id"].isin(fit_queries | audit | fresh_queries)]
frame = pd.concat([validation, test], ignore_index=True)

TOKEN = re.compile(r"[^\W_]+")
SENTENCE = re.compile(r"(?<=[.!?])\s+")

def tokens(text):
    return TOKEN.findall(text.casefold().replace("ё", "е"))

def query_window(query, text, limit):
    if len(text) <= limit:
        return text
    terms = {word[:5] for word in tokens(query) if len(word) >= 3}
    if not terms:
        return text[:limit]
    sentences = SENTENCE.split(text)
    offsets = np.cumsum([0] + [len(sentence) + 1 for sentence in sentences[:-1]])
    candidates = [min(max(0, int(offset) - limit // 4), len(text) - limit) for offset in offsets]
    candidates.append(len(text) - limit)
    windows = [text[start:start + limit] for start in candidates]
    return max(windows, key=lambda window: len(terms.intersection(word[:5] for word in tokens(window))))


original_graph = pd.read_csv(next(root.rglob("full_graph_scores.csv")))
graph = pd.concat([original_graph, fresh], ignore_index=True).set_index("id", verify_integrity=True)
base = frame["id"].map(graph["base"])
win = frame["id"].map(graph["window"])
full_score = frame["id"].map(graph["full"])
assert np.isfinite(pd.concat([base, win, full_score], axis=1).to_numpy()).all()
full = frame.copy()
full["title_clean"] = full["title_clean"].map(repair_lookalikes)
full["text_clean"] = [
    query_window(query, clean_text(repair_lookalikes(repair_encoding(text))), 1800)
    for query, text in zip(full["query"], full["text"])
]

def group_z(values):
    group = values.groupby(frame["query_id"])
    return (values - group.transform("mean")) / group.transform("std").fillna(0).clip(lower=1e-6)

semantic = pd.concat([cached(filename) for filename in (
    "bge_scores.npz", "horizon_scores.npz", "horizon_base_scores.npz", "lamar_scores.npz"
)], axis=1)
assert semantic.index.is_unique
features = pd.DataFrame({"base": base, "window": win, "full": full_score, "title_empty": frame["title_clean"].eq("")})
for name in semantic:
    feature = frame["id"].map(semantic[name])
    features[name] = group_z(feature)
    features[f"{name}_raw"] = feature
for name, value in (("full_minus_window", full_score - win), ("full_minus_base", full_score - base)):
    features[name] = value
features["words"] = full["text_clean"].map(lambda text: len(tokens(text)))
features["title_overlap"] = [
    len(set(tokens(q)).intersection(tokens(t))) for q, t in zip(full["query"], full["title_clean"])
]

def predict(query_ids):
    training = frame["query_id"].isin(query_ids)
    model = CatBoostClassifier(iterations=240, depth=3, learning_rate=.04, l2_leaf_reg=6,
                               random_seed=42, verbose=False, thread_count=4, allow_writing_files=False)
    model.fit(features.loc[training], frame.loc[training, "relevance"].astype(int))
    stack = group_z(pd.Series(model.predict_proba(features)[:, 1], index=frame.index))
    return pd.DataFrame({
        "original": .30 * base + .20 * win + .22 * full_score + .28 * stack,
        "full_lean": .24 * base + .15 * win + .33 * full_score + .28 * stack,
        "full_half": .18 * base + .18 * win + .36 * full_score + .28 * stack,
    })

def evaluate(predictions, queries, name):
    records = []
    for query_id, group in frame[frame["query_id"].isin(queries)].groupby("query_id"):
        y = group["relevance"].astype(int).to_numpy()
        metrics = {
            c: float(ndcg_score([y], [predictions.loc[group.index, c]], k=10)) if len(y) > 1 else float(y[0] > 0)
            for c in predictions
        }
        records.append({"query_id": query_id, "source": group["ingest_source"].mode().iat[0], **metrics})
    report = pd.DataFrame(records)
    print(name, report[list(predictions)].mean().round(6).to_dict(), flush=True)
    print(report.groupby("source")[list(predictions)].mean().round(6), flush=True)
    report.to_csv(output_dir / f"{name}.csv", index=False)
    return report


In [ ]:
from catboost import CatBoostRanker, Pool

labeled = frame["relevance"].notna()
old_qwen = pd.read_csv(next(root.rglob("qwen8_validation_scores.csv"))).set_index("id", verify_integrity=True)["qwen8"]
all_qwen = pd.concat([old_qwen, fresh.set_index("id")["qwen8"]], verify_integrity=True)
qwen = frame["id"].map(all_qwen)
previous = pd.read_csv(next(root.rglob("qwen8_submission.csv"))).set_index("id", verify_integrity=True)["prediction"]
old_jina = pd.read_csv(next(root.rglob("jina_full_half.csv"))).set_index("id", verify_integrity=True)["prediction"]
deployed = frame["id"].map(previous)
qwen_test = (deployed - .8 * group_z(frame["id"].map(old_jina))) / .2
qwen = qwen.combine_first(qwen_test)
assert np.isfinite(qwen).all()
matrix = features.assign(
    qwen=qwen,
    qwen_rank=qwen.groupby(frame["query_id"]).rank(pct=True),
    qwen_gap=qwen.groupby(frame["query_id"]).transform("max") - qwen,
    jina_qwen_agreement=qwen - .25 * base - .25 * win - .50 * full_score,
    jina_views_spread=features[["base", "window", "full"]].std(axis=1),
)

public_best = pd.read_csv(next(root.rglob("robust_rank_submission.csv"))).set_index("id", verify_integrity=True)["prediction"]
public_best = frame["id"].map(public_best)
frozen = group_z(.8 * group_z(.25 * base + .25 * win + .50 * full_score) + .2 * qwen)

def focused_pool(indices):
    sampled = frame.loc[indices, ["query_id", "relevance"]].reset_index(drop=True)
    strength = frozen.loc[indices].to_numpy()
    labels = sampled["relevance"].to_numpy()
    pairs, pair_weights = [], []
    for group in sampled.groupby("query_id", sort=False).indices.values():
        relevant, irrelevant = group[labels[group] == 1], group[labels[group] == 0]
        if not len(relevant) or not len(irrelevant):
            continue
        actual = group[np.argsort(-strength[group], kind="stable")]
        position = {index: rank for rank, index in enumerate(actual)}
        discount = {index: 1 / np.log2(rank + 2) if rank < 10 else 0 for index, rank in position.items()}
        positive, negative = np.meshgrid(relevant, irrelevant, indexing="ij")
        positive, negative = positive.ravel(), negative.ravel()
        gain = np.array([abs(discount[a] - discount[b]) for a, b in zip(positive, negative)])
        pair_weight = .05 + gain * (.25 + .75 * np.exp(-np.abs(strength[positive] - strength[negative])))
        pair_weights.extend(pair_weight / pair_weight.sum())
        pairs.extend(zip(positive.tolist(), negative.tolist()))
    return Pool(matrix.loc[indices], label=sampled["relevance"].astype(int),
                group_id=sampled["query_id"], pairs=pairs, pairs_weight=pair_weights,
                baseline=2 * strength)

def run_model(queries):
    ordered = frame.loc[frame["query_id"].isin(queries)].sort_values(["query_id", "id"]).index
    groups = frame.loc[ordered, "query_id"]
    labels = frame.loc[ordered, "relevance"].astype(int)
    champion_model = CatBoostRanker(iterations=320, depth=3, learning_rate=.035, l2_leaf_reg=8,
                                    loss_function="PairLogit", random_seed=42, thread_count=4,
                                    verbose=False, allow_writing_files=False)
    champion_model.fit(Pool(matrix.loc[ordered], label=labels, group_id=groups))
    rank = group_z(pd.Series(champion_model.predict(matrix), index=frame.index))
    deployed_val = deployed.combine_first(.8 * group_z(predict(queries)["full_half"]) + .2 * qwen)
    champion = group_z(public_best.combine_first(.65 * group_z(deployed_val) + .35 * rank))

    extended = frame.loc[frame["query_id"].isin(queries | audit)].sort_values(["query_id","id"]).index
    residual = CatBoostRanker(iterations=360, depth=3, learning_rate=.035, l2_leaf_reg=8,
                               loss_function="PairLogit", random_seed=42, thread_count=4,
                               verbose=False, allow_writing_files=False)
    residual.fit(focused_pool(extended))
    corrections = pd.Series(residual.predict(matrix), index=frame.index)
    champion_order = champion.groupby(frame["query_id"]).rank(ascending=False, method="first")
    leading = champion.where(champion_order.eq(1)).groupby(frame["query_id"]).transform("max")
    next_best = champion.where(champion_order.eq(2)).groupby(frame["query_id"]).transform("max")
    ambiguity = (1 - (leading - next_best).fillna(2) / 1.25).clip(lower=0, upper=1)
    medium = champion + .30 * ambiguity * corrections
    return pd.DataFrame({"champion": champion, "gated_medium": medium})


In [ ]:
reports = [
    evaluate(run_model(fit_queries - queries)[["champion", "gated_medium"]], queries, f"gated_seed_{seed}")
    for seed, queries in folds.items()
]
metrics = pd.DataFrame([r.mean(numeric_only=True) for r in reports])
deltas = metrics.drop(columns="champion").subtract(metrics["champion"], axis=0)
chosen = "gated_medium"
pred = run_model(fit_queries)
report = evaluate(pred[["champion", chosen]], fresh_queries, "gated_fresh_audit")
articles = set(frame.loc[frame["query_id"].isin(fit_queries | audit), "article_id"])
novel = {q for q, rows in frame[frame["query_id"].isin(fresh_queries)].groupby("query_id")
         if set(rows["article_id"]).isdisjoint(articles)}
gains = report[chosen] - report["champion"]
novel_gain = gains[report["query_id"].isin(novel)].mean()
print("Selected", chosen, "worst fold", deltas[chosen].min(), "fresh", gains.mean(),
      "novel count and delta", len(novel), novel_gain, flush=True)
sample = pd.read_csv(data / "sample_submission.csv")
values = sample["id"].map(pd.Series(pred[chosen].to_numpy(), index=frame["id"]))
assert np.isfinite(values).all()
submission_path = output_dir / "submission.csv"
sample.assign(prediction=values).to_csv(submission_path, index=False)
print("Сабмит сохранён:", submission_path)


import hashlib

test_queries = test.set_index("id", verify_integrity=True)["query_id"]
ranks = values.groupby(sample["id"].map(test_queries)).rank(method="min", ascending=False)
assert len(sample) == 13507 and sample["id"].is_unique and ranks.notna().all()
rank_csv = sample.assign(rank=ranks.astype(int))[["id", "rank"]].to_csv(index=False, lineterminator="\n")
rank_hash = hashlib.sha256(rank_csv.encode("utf-8")).hexdigest()
expected_rank_hash = "051c464ecf5e47603024f3646e349456b61fefe903ba2984d86d7a601168225c"
print("SHA-256 рангов:", rank_hash)
print("Ранжирование совпало с исходным сабмитом:", rank_hash == expected_rank_hash)
